## Computer Use with Qwen3-VL

This notebook demonstrates how to use Qwen3-VL for computer use. It takes a screenshot of a user's desktop and a query, and then uses the model to interpret the user's query on the screenshot.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
# Required: OPENAI_API_KEY, OPENAI_BASE_URL, MODEL_ID
load_dotenv()

# Set OpenAI API credentials (required for OpenAI-compatible server)
API_KEY = os.environ["OPENAI_API_KEY"]
BASE_URL = os.environ["OPENAI_BASE_URL"]
MODEL_ID = os.environ["MODEL_ID"]

print("✓ Environment variables loaded successfully")
print(f"  API_KEY: {API_KEY[:8]}..." if API_KEY else "  API_KEY: (empty)")
print(f"  BASE_URL: {BASE_URL}")
print(f"  MODEL_ID: {MODEL_ID}")


### \[Setup\]

Load visualization utils.

In [ ]:
%pip install git+https://github.com/huggingface/transformers
%pip install qwen-vl-utils
%pip install qwen-agent
%pip install openai

In [ ]:
from PIL import Image, ImageDraw, ImageColor

def draw_point(image: Image.Image, point: list, color=None):
    if isinstance(color, str):
        try:
            color = ImageColor.getrgb(color)
            color = color + (128,)  
        except ValueError:
            color = (255, 0, 0, 128)  
    else:
        color = (255, 0, 0, 128)  

    overlay = Image.new('RGBA', image.size, (255, 255, 255, 0))
    overlay_draw = ImageDraw.Draw(overlay)
    radius = min(image.size) * 0.05
    x, y = point 

    overlay_draw.ellipse(
        [(x - radius, y - radius), (x + radius, y + radius)],
        fill=color
    )
    
    center_radius = radius * 0.1
    overlay_draw.ellipse(
        [(x - center_radius, y - center_radius), 
         (x + center_radius, y + center_radius)],
        fill=(0, 255, 0, 255)
    )

    image = image.convert('RGBA')
    combined = Image.alpha_composite(image, overlay)

    return combined.convert('RGB')

def smart_resize(height, width, factor=32, min_pixels=65536, max_pixels=2457600):
    if height < factor or width < factor:
        raise ValueError(f"La imagen es demasiado pequeña: {height}x{width}")
    
    # Calcular escala para mantenerse dentro de los límites de píxeles
    h_w_ratio = height / width
    target_pixels = max(min(height * width, max_pixels), min_pixels)
    
    new_h = int((target_pixels * h_w_ratio)**0.5)
    new_w = int((target_pixels / h_w_ratio)**0.5)
    
    # Ajustar al múltiplo más cercano del factor (32 para Qwen3.5)
    new_h = max(round(new_h / factor) * factor, factor)
    new_w = max(round(new_w / factor) * factor, factor)
    
    return new_h, new_w

#### Computer Use with Inference API



In [ ]:
import os
import json
import base64
from openai import OpenAI
from PIL import Image
from IPython.display import display
from qwen_agent.llm.fncall_prompts.nous_fncall_prompt import (
    NousFnCallPrompt,
    Message,
    ContentItem,
)

from utils.agent_function_call import ComputerUse

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def perform_gui_grounding_with_api(screenshot_path, user_query, model_id, min_pixels=3136, max_pixels=12845056):
    """
    Perform GUI grounding using Qwen model to interpret user query on a screenshot.
    
    Args:
        screenshot_path (str): Path to the screenshot image
        user_query (str): User's query/instruction
        model: Preloaded Qwen model
        min_pixels: Minimum pixels for the image
        max_pixels: Maximum pixels for the image
        
    Returns:
        tuple: (output_text, display_image) - Model's output text and annotated image
    """

    # Open and process image
    input_image = Image.open(screenshot_path)
    base64_image = encode_image(screenshot_path)
    client = OpenAI(
        #If the environment variable is not configured, please replace the following line with the Dashscope API Key: api_key="sk-xxx". Access via https://bailian.console.alibabacloud.com/?apiKey=1 "
        api_key=API_KEY,
        base_url=BASE_URL,
    )
    resized_height, resized_width = smart_resize(
        input_image.height,
        input_image.width,
        factor=32,
        min_pixels=min_pixels,
        max_pixels=max_pixels,
    )
    
    # Initialize computer use function
    computer_use = ComputerUse(
        cfg={"display_width_px": 1000, "display_height_px": 1000}
    )

    # Build messages
    system_message = NousFnCallPrompt().preprocess_fncall_messages(
        messages=[
            Message(role="system", content=[ContentItem(text="You are a helpful assistant.")]),
        ],
        functions=[computer_use.function],
        lang=None,
    )
    system_message = system_message[0].model_dump()
    messages=[
        {
            "role": "system",
            "content": [
                {"type": "text", "text": msg["text"]} for msg in system_message["content"]
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    # "min_pixels": 1024,
                    # "max_pixels": max_pixels,
                    # Pass in BASE64 image data. Note that the image format (i.e., image/{format}) must match the Content Type in the list of supported images. "f" is the method for string formatting.
                    # PNG image:  f"data:image/png;base64,{base64_image}"
                    # JPEG image: f"data:image/jpeg;base64,{base64_image}"
                    # WEBP image: f"data:image/webp;base64,{base64_image}"
                    "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"},
                },
                {"type": "text", "text": user_query},
            ],
        }
    ]
    #print(json.dumps(messages, indent=4))
    completion = client.chat.completions.create(
        model = model_id,
        messages = messages,
       
    )
    
    output_text = completion.choices[0].message.content
    

    # Parse action and visualize
    action = json.loads(output_text.split('<tool_call>\n')[1].split('\n</tool_call>')[0])
    coordinate_relative = action['arguments']['coordinate']
    coordinate_absolute = [coordinate_relative[0] / 1000 * resized_width, coordinate_relative[1] / 1000 * resized_height]

    display_image = input_image.resize((resized_width, resized_height))
    display_image = draw_point(display_image, coordinate_absolute, color='green')
    
    return output_text, display_image

In [ ]:
# Example usage
screenshot = "assets/computer_use/computer_use2.jpeg"
user_query = 'open the second issue'
model_id = MODEL_ID
output_text, display_image = perform_gui_grounding_with_api(screenshot, user_query, model_id)

# Display results
#print(output_text)
display(display_image)

#### Computer Use with Local Model



In [ ]:
import json
from PIL import Image
from IPython.display import display
from qwen_agent.llm.fncall_prompts.nous_fncall_prompt import (
    NousFnCallPrompt,
    Message,
    ContentItem,
)
from utils.agent_function_call import ComputerUse


In [ ]:
# Example usage
screenshot = "assets/computer_use/computer_use1.jpeg"
user_query = 'Reload web page'
model_id = MODEL_ID
output_text, display_image = perform_gui_grounding_with_api(screenshot, user_query, model_id)


# Display results
print(output_text)
display(display_image)

In [ ]:
# Example usage
screenshot = "assets/computer_use/computer_use2.jpeg"
user_query = 'open the first issue'
model_id = MODEL_ID
output_text, display_image = perform_gui_grounding_with_api(screenshot, user_query, model_id)

# Display results
print(output_text)
display(display_image)